In [1]:
%cd ..
import json
import os
import subprocess

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope
IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/work/dlclarge2/aliy-vjepa/vjepa2


/work/dlclarge2/aliy-maskgit/miniconda3/envs/vjepa2/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
def forward_target(c):
    with torch.no_grad():
        h = target_encoder(c)
        h = [F.layer_norm(hi, (hi.size(-1),)) for hi in h]
        return h

def forward_context(c):
    z = encoder(c, masks_enc)
    z = predictor(z, masks_enc, masks_pred)
    return z

def loss_fn(z, h):
    # Assumption: predictor will have returned only masked tokens for z
    h = [apply_masks(hi, mi, concat=False) for hi, mi in zip(h, masks_pred)]

    loss, n = 0, 0
    for zi, hi in zip(z, h):
        for zij, hij in zip(zi, hi):
            loss += torch.mean(torch.abs(zij - hij) ** loss_exp) / loss_exp
            n += 1
    loss /= n
    return loss

In [2]:
def get_video():
    vr = VideoReader("00a0f008-a315437f.mov")
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, 128, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video


def forward_vjepa_video(model_hf, hf_transform):
    # Run a sample inference with VJEPA
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        #x_pt = pt_transform(video).cuda().unsqueeze(0)
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")
        print(f"x_hf shape: {x_hf.shape}")
        # Extract the patch-wise features from the last layer
        #out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)
        output_predictor = model_hf.predictor(out_patch_features_hf)
    return out_patch_features_hf, output_predictor #, out_patch_features_pt




In [3]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-256"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.


In [ ]:
def load_clips():
    all_clips, all_masks_enc, all_masks_pred = [], [], []
    for fpc_sample in sample:
        udata, masks_enc, masks_pred = fpc_sample
        all_clips += [udata[0][0].to(device, non_blocking=True)]
        all_masks_enc += [[m.to(device, non_blocking=True) for m in masks_enc]]
        all_masks_pred += [[m.to(device, non_blocking=True) for m in masks_pred]]
    return all_clips, all_masks_enc, all_masks_pred

clips, masks_enc, masks_pred = load_clips()

In [ ]:
def forward_target(c):
    with torch.no_grad():
        h = model_hf.target_encoder(c)
        h = [F.layer_norm(hi, (hi.size(-1),)) for hi in h]
        return h

def forward_context(c):
    z = model_hf.encoder(c, masks_enc)
    z = model_hf.predictor(z, masks_enc, masks_pred)
    return z

In [ ]:

# Inference on video to get the patch-wise features
out_patch_features_hf, pp = forward_vjepa_video(model_hf, hf_transform)

print(
    f"""
    Inference results on video:
    HuggingFace output shape: {out_patch_features_hf.shape}
    """
)

x_hf shape: torch.Size([1, 64, 3, 256, 256])


TypeError: VJEPA2Predictor.forward() missing 2 required positional arguments: 'context_mask' and 'target_mask'

In [11]:
model_hf.predictor

VJEPA2Predictor(
  (embeddings): VJEPA2PredictorEmbeddings(
    (predictor_embeddings): Linear(in_features=1408, out_features=384, bias=True)
  )
  (layer): ModuleList(
    (0-11): 12 x VJEPA2Layer(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attention): VJEPA2RopeAttention(
        (query): Linear(in_features=384, out_features=384, bias=True)
        (key): Linear(in_features=384, out_features=384, bias=True)
        (value): Linear(in_features=384, out_features=384, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): VJEPA2MLP(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (activation): GELUActivation()
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
      )
    )
  )
  (layernorm): LayerNorm((384,), eps=1e-

In [1]:
import torch

# preprocessor
processor = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_preprocessor')
# models
vjepa2_vit_giant = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant')


Downloading: "https://github.com/facebookresearch/vjepa2/zipball/main" to /home/aliy/.cache/torch/hub/main.zip


Using cache found in /home/aliy/.cache/torch/hub/facebookresearch_vjepa2_main
/work/dlclarge2/aliy-maskgit/miniconda3/envs/vjepa2/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Downloading: "https://dl.fbaipublicfiles.com/vjepa2/vitg.pt" to /home/aliy/.cache/torch/hub/checkpoints/vitg.pt


100%|██████████| 15.3G/15.3G [10:03<00:00, 27.3MB/s]  


In [2]:
%cd ..

/work/dlclarge2/aliy-vjepa/vjepa2


In [4]:
#load a pt file
checkpoint = "/home/aliy/.cache/torch/hub/checkpoints/vitg.pt"  # Replace with your checkpoint path
checkpoint = torch.load(checkpoint, map_location="cpu")
# Load the model state dict
#vjepa2_vit_giant.load_state_dict(checkpoint['model'], strict=False) 


In [6]:
checkpoint.keys()

dict_keys(['encoder', 'predictor', 'opt', 'scaler', 'target_encoder', 'epoch', 'loss', 'batch_size', 'world_size', 'lr'])